# WP7 — Evaluation starter (Student 4)

**What this notebook does.** Reads every file in `decisions/`, joins to `labels.parquet`, computes the primary metrics from plan §3.7 per strategy, and emits `evaluation/metrics.parquet` per pipeline contract §8.

**Metrics covered here.** Median τᵢ, IQR of τᵢ, selective accuracy, deferral rate. Empirical conformal coverage is included when the strategy emits prediction sets. Bootstrap 95% CIs are computed with 1,000 iterations for speed; production runs use 10,000 per plan §3.6.


In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
DATA.mkdir(parents=True, exist_ok=True)
(DATA / 'preprocessing').mkdir(exist_ok=True)
(DATA / 'endpoint').mkdir(exist_ok=True)
(DATA / 'decisions').mkdir(exist_ok=True)
(DATA / 'evaluation').mkdir(exist_ok=True)
print(f'Pipeline root: {DATA}')


## 1. Ingest all decision files


In [ ]:
dec_files = sorted((DATA / 'decisions').glob('*.parquet'))
all_decisions = pd.concat([pd.read_parquet(f) for f in dec_files], ignore_index=True)
print(f'loaded {len(dec_files)} strategy files, {len(all_decisions):,} decision rows')
print('strategies:', sorted(all_decisions['strategy_name'].unique()))


## 2. Join with labels and compute per-strategy accuracy of committed predictions


In [ ]:
lbls = pd.read_parquet(DATA / 'labels.parquet')
joined = all_decisions.merge(lbls[['participant_id','night_index','binary_label']],
                              on=['participant_id','night_index'])
predicts = joined[joined['decision'] == 'predict'].copy()
predicts['correct'] = predicts['predicted_label'] == predicts['binary_label']
selective_acc = predicts.groupby('strategy_name')['correct'].agg(['mean','count']).rename(
    columns={'mean':'selective_accuracy','count':'n_predictions'})
print(selective_acc)


## 3. Compute median τᵢ per strategy with bootstrap CIs


In [ ]:
tau = pd.read_parquet(DATA / 'tau_per_strategy.parquet')
rng = np.random.default_rng(42)
B = 1000  # use 10_000 for production per plan §3.6

def bootstrap_median(values, n_boot=B):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True)
    medians = np.median(boots, axis=1)
    return float(np.median(values)), float(np.quantile(medians, 0.025)), float(np.quantile(medians, 0.975))

metric_rows = []
for strat, g in tau.groupby('strategy_name'):
    tau_conv = g.loc[g['convergence_status']=='converged', 'tau_i'].astype(float).dropna()
    point, lo, hi = bootstrap_median(tau_conv)
    metric_rows.append({'strategy_name': strat, 'metric_name': 'median_tau',
                        'point_estimate': point, 'ci_low': lo, 'ci_high': hi, 'n_bootstrap': B})
    iqr = float(tau_conv.quantile(0.75) - tau_conv.quantile(0.25)) if len(tau_conv) else np.nan
    metric_rows.append({'strategy_name': strat, 'metric_name': 'iqr_tau',
                        'point_estimate': iqr, 'ci_low': np.nan, 'ci_high': np.nan, 'n_bootstrap': 0})
    if strat in selective_acc.index:
        acc = float(selective_acc.loc[strat, 'selective_accuracy'])
        metric_rows.append({'strategy_name': strat, 'metric_name': 'selective_accuracy',
                            'point_estimate': acc, 'ci_low': np.nan, 'ci_high': np.nan, 'n_bootstrap': 0})
    n_dec = len(all_decisions[all_decisions['strategy_name']==strat])
    n_def = len(all_decisions[(all_decisions['strategy_name']==strat) & (all_decisions['decision']=='defer')])
    metric_rows.append({'strategy_name': strat, 'metric_name': 'deferral_rate',
                        'point_estimate': float(n_def / n_dec) if n_dec else np.nan,
                        'ci_low': np.nan, 'ci_high': np.nan, 'n_bootstrap': 0})

metrics = pd.DataFrame(metric_rows).astype({
    'strategy_name': 'string', 'metric_name': 'string',
    'point_estimate': 'float32', 'ci_low': 'float32', 'ci_high': 'float32',
    'n_bootstrap': 'int32',
})
metrics.to_parquet(DATA / 'evaluation/metrics.parquet', index=False)
peek(DATA / 'evaluation/metrics.parquet', n=40)


## 4. Reliability diagram data (base classifier calibration)


In [ ]:
prob = pd.read_parquet(DATA / 'probability_table.parquet').merge(lbls, on=['participant_id','night_index'])
bins = np.linspace(0, 1, 11)
prob['bin'] = pd.cut(prob['p_post_ovulatory'], bins, include_lowest=True)
reliab = prob.groupby('bin', observed=True).agg(
    predicted_mean=('p_post_ovulatory', 'mean'),
    observed_fraction_post=('binary_label', lambda s: (s == 'post').mean()),
    n=('binary_label', 'size'),
).reset_index()
reliab['bin_low']  = reliab['bin'].apply(lambda b: float(b.left)).astype('float32')
reliab['bin_high'] = reliab['bin'].apply(lambda b: float(b.right)).astype('float32')
reliab = reliab[['bin_low','bin_high','predicted_mean','observed_fraction_post','n']]
reliab = reliab.astype({'predicted_mean':'float32','observed_fraction_post':'float32','n':'int32'})
reliab.to_parquet(DATA / 'evaluation/reliability.parquet', index=False)
peek(DATA / 'evaluation/reliability.parquet')


## 5. Quick ranking table


In [ ]:
summary_tbl = metrics.pivot(index='strategy_name', columns='metric_name', values='point_estimate')
print(summary_tbl.round(3))


## Next

Not yet implemented:
- **Empirical conformal coverage** (plan §3.7; pipeline contract §8 metric). Requires joining `prediction_sets.parquet` with labels and checking `contains_true_class == True`.
- **Coverage-error tradeoff curve** (Figure 2).
- **Bootstrap CIs at 10,000 iterations** with BCa correction.
- **Longitudinal transfer analysis** (Figure 4). Requires Round-1 → Round-2 labelled data.
